# 1 EDA - Dataset limpo UFC

Objetivo: olhar o dataset final salvo em `ufc-master_more_clean.csv`, entender quantos dados sobraram e listar quais features estao disponiveis para a proxima etapa de EDA/modelagem.

In [2]:
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

csv_path = Path('ufc-master_more_clean.csv')
df = pd.read_csv(csv_path, parse_dates=['date'])

print(f'Dataset: {csv_path.name}')
print(f'Linhas: {df.shape[0]:,}')
print(f'Colunas: {df.shape[1]:,}')

df.head()

Dataset: ufc-master_more_clean.csv
Linhas: 2,625
Colunas: 47


,date,R_fighter,B_fighter,Winner,weight_class,gender,no_of_rounds,R_current_win_streak,B_current_win_streak,R_longest_win_streak,B_longest_win_streak,R_current_lose_streak,B_current_lose_streak,R_wins,B_wins,R_losses,B_losses,R_total_rounds_fought,B_total_rounds_fought,R_total_title_bouts,B_total_title_bouts,R_win_by_KO/TKO,B_win_by_KO/TKO,R_win_by_Submission,B_win_by_Submission,R_Height_cms,B_Height_cms,R_Reach_cms,B_Reach_cms,R_age,B_age,R_avg_SIG_STR_landed,B_avg_SIG_STR_landed,R_avg_SIG_STR_pct,B_avg_SIG_STR_pct,R_avg_SUB_ATT,B_avg_SUB_ATT,R_avg_TD_landed,B_avg_TD_landed,R_avg_TD_pct,B_avg_TD_pct,R_Stance,B_Stance,R_odds,B_odds,R_ufc_fights,B_ufc_fights
0,2026-03-28,Israel Adesanya,Joe Pyfer,Blue,Middleweight,MALE,5,0,3,9,4,3,0,13,7,5,2,66,18,12,0,5,4,0,2,193.04,187.96,203.20,190.50,36,29,4.03,3.52,0.48,0.44,0.1,0.9,0.05,1.45,0.09,0.30,Switch,Orthodox,-130.0,102.0,18,9
1,2026-03-28,Michael Chiesa,Niko Price,Red,Welterweight,MALE,3,3,0,4,2,0,3,14,8,7,10,47,40,1,0,0,4,8,2,185.42,182.88,190.50,193.04,38,36,2.02,5.11,0.40,0.43,1.0,0.6,3.11,1.06,0.47,0.30,Southpaw,Orthodox,-901.0,550.0,21,18
2,2026-03-28,Mansur Abdul-Malik,Yousri Belgaroui,Blue,Middleweight,MALE,3,4,2,3,2,0,0,4,2,0,1,9,9,0,0,3,2,1,0,187.96,198.12,203.20,200.66,28,33,3.28,6.10,0.44,0.64,0.3,0.0,1.65,0.29,0.41,1.00,Orthodox,Orthodox,-158.0,124.0,4,3
3,2026-03-28,Terrance McKinney,Kyle Nelson,Red,Lightweight,MALE,3,0,1,2,3,1,0,7,5,6,5,16,26,0,0,4,2,3,0,177.80,180.34,185.42,180.34,31,34,6.53,3.60,0.55,0.45,2.1,0.5,3.31,1.18,0.40,0.23,Switch,Switch,-150.0,118.0,13,10
4,2026-03-28,Navajo Stirling,Bruno Lopes,Red,Light Heavyweight,MALE,3,4,0,4,2,0,1,4,2,0,2,11,7,0,0,1,1,0,0,193.04,187.96,200.66,187.96,28,32,6.25,2.88,0.52,0.43,0.0,0.0,0.98,1.93,0.28,0.21,Orthodox,Orthodox,-440.0,310.0,4,4


## 1.1 O que sobrou no dataset?

Esse CSV representa o recorte mais limpo ate agora: lutas masculinas, categorias escolhidas, periodo moderno e ambos os lutadores com pelo menos 2 lutas previas no UFC.

In [3]:
resumo_dataset = pd.DataFrame({
    'metrica': [
        'linhas',
        'colunas',
        'data_inicial',
        'data_final',
        'categorias_de_peso',
        'lutadores_unicos',
    ],
    'valor': [
        len(df),
        df.shape[1],
        df['date'].min().date(),
        df['date'].max().date(),
        df['weight_class'].nunique(),
        pd.concat([df['R_fighter'], df['B_fighter']]).nunique(),
    ]
})

resumo_dataset

,metrica,valor
0,linhas,2625
1,colunas,47
2,data_inicial,2015-01-03
3,data_final,2026-03-28
4,categorias_de_peso,6
5,lutadores_unicos,1019


## 1.2 Features disponiveis

Abaixo esta a lista completa das colunas que sobraram, com indice para facilitar referencia durante a EDA.

In [4]:
features = pd.DataFrame({
    'idx': range(len(df.columns)),
    'feature': df.columns,
    'tipo': df.dtypes.astype(str).values,
    'qtd_faltante': df.isna().sum().values,
    'pct_faltante': (df.isna().mean().values * 100).round(2),
})

features

,idx,feature,tipo,qtd_faltante,pct_faltante
0,0,date,datetime64[us],0,0.00
1,1,R_fighter,str,0,0.00
2,2,B_fighter,str,0,0.00
3,3,Winner,str,0,0.00
4,4,weight_class,str,0,0.00
5,5,gender,str,0,0.00
6,6,no_of_rounds,int64,0,0.00
7,7,R_current_win_streak,int64,0,0.00
8,8,B_current_win_streak,int64,0,0.00
9,9,R_longest_win_streak,int64,0,0.00


## 1.3 Features por grupo

Separando as colunas por papel ajuda a decidir o que pode entrar como variavel explicativa e o que deve ficar fora do modelo.

In [5]:
colunas_identificacao = ['date', 'R_fighter', 'B_fighter']
target = ['Winner']
colunas_contexto = ['weight_class', 'gender', 'no_of_rounds']
colunas_blue = [col for col in df.columns if col.startswith('B_')]
colunas_red = [col for col in df.columns if col.startswith('R_')]
colunas_experiencia = ['R_ufc_fights', 'B_ufc_fights']

grupos_features = {
    'identificacao': colunas_identificacao,
    'target': target,
    'contexto_da_luta': colunas_contexto,
    'blue_corner': colunas_blue,
    'red_corner': colunas_red,
    'experiencia_ufc': colunas_experiencia,
}

for grupo, colunas in grupos_features.items():
    print(f'\n{grupo} ({len(colunas)} colunas)')
    display(pd.DataFrame({'feature': colunas}))


identificacao (3 colunas)


,feature
0,date
1,R_fighter
2,B_fighter



target (1 colunas)


,feature
0,Winner



contexto_da_luta (3 colunas)


...

,feature
0,B_fighter
1,B_current_win_streak
2,B_longest_win_streak
3,B_current_lose_streak
4,B_wins
5,B_losses
6,B_total_rounds_fought
7,B_total_title_bouts
8,B_win_by_KO/TKO
9,B_win_by_Submission



red_corner (21 colunas)


,feature
0,R_fighter
1,R_current_win_streak
2,R_longest_win_streak
3,R_current_lose_streak
4,R_wins
5,R_losses
6,R_total_rounds_fought
7,R_total_title_bouts
8,R_win_by_KO/TKO
9,R_win_by_Submission



experiencia_ufc (2 colunas)


,feature
0,R_ufc_fights
1,B_ufc_fights


## 1.4 Recortes rapidos

Conferencias basicas para entender a distribuicao do dataset final.

In [6]:
recortes = {
    'vencedor_red_blue': df['Winner'].value_counts(dropna=False),
    'genero': df['gender'].value_counts(dropna=False),
    'categorias': df['weight_class'].value_counts(dropna=False),
    'rounds_previstos': df['no_of_rounds'].value_counts(dropna=False).sort_index(),
    'stance_red': df['R_Stance'].value_counts(dropna=False),
    'stance_blue': df['B_Stance'].value_counts(dropna=False),
    'years': df['date'].dt.year.value_counts(dropna=False).sort_index(),
}

for nome, serie in recortes.items():
    print(f'\n{nome}')
    display(serie)


vencedor_red_blue


Winner
Red           1467
Blue          1152
Draw             4
No Contest       2
Name: count, dtype: int64


genero


gender
MALE    2625
Name: count, dtype: int64


categorias


...

R_Stance
Orthodox    1837
Southpaw     560
Switch       228
Name: count, dtype: int64


stance_blue


B_Stance
Orthodox    1802
Southpaw     588
Switch       235
Name: count, dtype: int64


years


date
2015    221
2016    242
2017    201
2018    219
2019    210
2020    175
2021    247
2022    255
2023    242
2024    275
2025    281
2026     57
Name: count, dtype: int64

## 1.5 Faltantes

Aqui vemos se ainda existem buracos depois dos filtros.

In [7]:
faltantes = (
    df.isna().sum()
    .sort_values(ascending=False)
    .rename('qtd_faltante')
    .to_frame()
)
faltantes['pct_faltante'] = (faltantes['qtd_faltante'] / len(df) * 100).round(2)

faltantes.query('qtd_faltante > 0')

,qtd_faltante,pct_faltante
B_odds,108,4.11
R_odds,108,4.11
B_avg_SIG_STR_landed,42,1.60
R_avg_SIG_STR_landed,42,1.60
R_avg_TD_pct,3,0.11
B_avg_TD_pct,3,0.11


## 1.6 Investigação da confiabilidade usando histórico de lutas de um atleta

In [8]:
adesanya_fights = df[
    (df['R_fighter'] == 'Israel Adesanya') |
    (df['B_fighter'] == 'Israel Adesanya')
]


charles_oliveira_fights = df[
    (df['R_fighter'] == 'Charles Oliveira') |   
    (df['B_fighter'] == 'Charles Oliveira')
]   


adesanya_fights

,date,R_fighter,B_fighter,Winner,weight_class,gender,no_of_rounds,R_current_win_streak,B_current_win_streak,R_longest_win_streak,B_longest_win_streak,R_current_lose_streak,B_current_lose_streak,R_wins,B_wins,R_losses,B_losses,R_total_rounds_fought,B_total_rounds_fought,R_total_title_bouts,B_total_title_bouts,R_win_by_KO/TKO,B_win_by_KO/TKO,R_win_by_Submission,B_win_by_Submission,R_Height_cms,B_Height_cms,R_Reach_cms,B_Reach_cms,R_age,B_age,R_avg_SIG_STR_landed,B_avg_SIG_STR_landed,R_avg_SIG_STR_pct,B_avg_SIG_STR_pct,R_avg_SUB_ATT,B_avg_SUB_ATT,R_avg_TD_landed,B_avg_TD_landed,R_avg_TD_pct,B_avg_TD_pct,R_Stance,B_Stance,R_odds,B_odds,R_ufc_fights,B_ufc_fights
0,2026-03-28,Israel Adesanya,Joe Pyfer,Blue,Middleweight,MALE,5,0,3,9,4,3,0,13,7,5,2,66,18,12,0,5,4,0,2,193.04,187.96,203.20,190.50,36,29,4.0300,3.5200,0.480,0.440,0.1000,0.9000,0.0500,1.4500,0.090,0.300,Switch,Orthodox,-130.0,102.0,18,9
318,2025-02-01,Israel Adesanya,Nassourdine Imavov,Blue,Middleweight,MALE,5,0,3,9,3,2,0,13,7,4,2,64,32,12,0,5,3,0,0,193.04,190.50,203.20,190.50,35,28,4.0300,4.2800,0.480,0.540,0.1000,1.0000,0.0500,0.7400,0.090,0.320,Switch,Orthodox,-158.0,134.0,17,9
430,2024-08-17,Dricus Du Plessis,Israel Adesanya,Red,Middleweight,MALE,5,7,0,7,9,0,1,7,13,0,3,18,60,1,11,4,5,1,0,185.42,193.04,193.04,203.20,30,35,6.1800,4.0000,0.490,0.480,0.9000,0.1000,3.0400,0.0500,0.500,0.120,Switch,Switch,-120.0,100.0,7,16
679,2023-09-09,Israel Adesanya,Sean Strickland,Blue,Middleweight,MALE,5,1,2,9,6,0,0,13,14,2,5,55,56,11,1,5,4,0,1,193.04,185.42,203.20,193.04,34,32,4.0000,6.0100,0.480,0.420,0.1000,0.2000,0.0500,0.7800,0.120,0.640,Switch,Orthodox,-550.0,410.0,15,19
800,2023-04-08,Alex Pereira,Israel Adesanya,Blue,Middleweight,MALE,5,4,0,4,9,0,1,4,12,0,2,11,53,4,11,3,4,0,0,193.04,193.04,200.66,203.20,35,33,5.2300,4.0000,0.630,0.480,0.3000,0.1000,0.1700,0.0500,1.000,0.120,Orthodox,Switch,NaN,NaN,4,14
884,2022-11-12,Israel Adesanya,Alex Pereira,Blue,Middleweight,MALE,5,3,3,9,3,0,0,12,3,1,0,48,6,11,4,4,2,0,0,193.04,193.04,203.20,200.66,33,35,4.0000,5.2300,0.480,0.630,0.1000,0.3000,0.0500,0.1700,0.120,1.000,Switch,Orthodox,-210.0,180.0,13,3
987,2022-07-02,Israel Adesanya,Jared Cannonier,Red,Middleweight,MALE,5,2,2,9,3,0,0,11,8,1,5,43,30,11,0,4,6,0,0,193.04,180.34,203.20,195.58,32,38,4.0000,4.4900,0.480,0.500,0.1000,0.0000,0.0500,0.4200,0.120,0.460,Switch,Switch,-540.0,420.0,12,13
1091,2022-02-12,Israel Adesanya,Robert Whittaker,Red,Middleweight,MALE,5,1,3,9,9,0,0,10,14,1,3,38,50,11,3,4,5,0,0,193.04,182.88,203.20,185.42,32,31,4.0000,4.5800,0.480,0.430,0.1000,0.0000,0.0500,0.8000,0.120,0.380,Switch,Orthodox,-310.0,245.0,11,17
1248,2021-06-12,Israel Adesanya,Marvin Vettori,Red,Middleweight,MALE,5,0,5,9,5,1,0,9,7,1,2,33,30,5,0,4,0,0,2,193.04,182.88,203.20,187.96,31,27,3.9400,3.8800,0.500,0.430,0.2000,0.7000,0.0000,2.2400,0.000,0.470,Switch,Southpaw,-235.0,185.0,10,9
1315,2021-03-06,Jan Blachowicz,Israel Adesanya,Red,Light Heavyweight,MALE,5,4,9,4,9,0,0,10,9,5,0,39,28,1,4,4,4,2,0,187.96,193.04,198.12,203.20,38,31,3.5900,3.9500,0.490,0.490,0.2000,0.3000,1.1800,0.0000,0.530,0.000,Orthodox,Switch,205.0,-245.0,15,9


In [9]:
red_stats = adesanya_fights[adesanya_fights['R_fighter'] == 'Israel Adesanya'].copy()
blue_stats = adesanya_fights[adesanya_fights['B_fighter'] == 'Israel Adesanya'].copy()

red_stats = red_stats.assign(
    side='R',
    fighter='Israel Adesanya',
    opponent=red_stats['B_fighter'],
)[
    [
        'date', 'side', 'fighter', 'opponent', 'weight_class', 'Winner',
        'R_age', 'R_Height_cms', 'R_Reach_cms',
        'R_avg_SIG_STR_landed', 'R_avg_SIG_STR_pct', 'R_avg_SUB_ATT',
        'R_avg_TD_landed', 'R_avg_TD_pct',
        'R_ufc_fights', 'R_current_win_streak', 'R_longest_win_streak',
        'R_current_lose_streak', 'R_wins', 'R_losses',
        'R_total_rounds_fought', 'R_total_title_bouts',
        'R_win_by_KO/TKO', 'R_win_by_Submission'
    ]
].rename(columns={
    'R_age': 'age',
    'R_Height_cms': 'height_cm',
    'R_Reach_cms': 'reach_cm',
    'R_avg_SIG_STR_landed': 'avg_SIG_STR_landed',
    'R_avg_SIG_STR_pct': 'avg_SIG_STR_pct',
    'R_avg_SUB_ATT': 'avg_SUB_ATT',
    'R_avg_TD_landed': 'avg_TD_landed',
    'R_avg_TD_pct': 'avg_TD_pct',
    'R_ufc_fights': 'ufc_fights',
    'R_current_win_streak': 'current_win_streak',
    'R_longest_win_streak': 'longest_win_streak',
    'R_current_lose_streak': 'current_lose_streak',
    'R_wins': 'wins',
    'R_losses': 'losses',
    'R_total_rounds_fought': 'total_rounds_fought',
    'R_total_title_bouts': 'total_title_bouts',
    'R_win_by_KO/TKO': 'win_by_KO/TKO',
    'R_win_by_Submission': 'win_by_Submission',
})

blue_stats = blue_stats.assign(
    side='B',
    fighter='Israel Adesanya',
    opponent=blue_stats['R_fighter'],
)[
    [
        'date', 'side', 'fighter', 'opponent', 'weight_class', 'Winner',
        'B_age', 'B_Height_cms', 'B_Reach_cms',
        'B_avg_SIG_STR_landed', 'B_avg_SIG_STR_pct', 'B_avg_SUB_ATT',
        'B_avg_TD_landed', 'B_avg_TD_pct',
        'B_ufc_fights', 'B_current_win_streak', 'B_longest_win_streak',
        'B_current_lose_streak', 'B_wins', 'B_losses',
        'B_total_rounds_fought', 'B_total_title_bouts',
        'B_win_by_KO/TKO', 'B_win_by_Submission'
    ]
].rename(columns={
    'B_age': 'age',
    'B_Height_cms': 'height_cm',
    'B_Reach_cms': 'reach_cm',
    'B_avg_SIG_STR_landed': 'avg_SIG_STR_landed',
    'B_avg_SIG_STR_pct': 'avg_SIG_STR_pct',
    'B_avg_SUB_ATT': 'avg_SUB_ATT',
    'B_avg_TD_landed': 'avg_TD_landed',
    'B_avg_TD_pct': 'avg_TD_pct',
    'B_ufc_fights': 'ufc_fights',
    'B_current_win_streak': 'current_win_streak',
    'B_longest_win_streak': 'longest_win_streak',
    'B_current_lose_streak': 'current_lose_streak',
    'B_wins': 'wins',
    'B_losses': 'losses',
    'B_total_rounds_fought': 'total_rounds_fought',
    'B_total_title_bouts': 'total_title_bouts',
    'B_win_by_KO/TKO': 'win_by_KO/TKO',
    'B_win_by_Submission': 'win_by_Submission',
})

adesanya_stats = pd.concat([red_stats, blue_stats], ignore_index=True).sort_values('date', ascending=False)
adesanya_stats

,date,side,fighter,opponent,weight_class,Winner,age,height_cm,reach_cm,avg_SIG_STR_landed,avg_SIG_STR_pct,avg_SUB_ATT,avg_TD_landed,avg_TD_pct,ufc_fights,current_win_streak,longest_win_streak,current_lose_streak,wins,losses,total_rounds_fought,total_title_bouts,win_by_KO/TKO,win_by_Submission
0,2026-03-28,R,Israel Adesanya,Joe Pyfer,Middleweight,Blue,36,193.04,203.2,4.03,0.480,0.1000,0.05,0.09,18,0,9,3,13,5,66,12,5,0
1,2025-02-01,R,Israel Adesanya,Nassourdine Imavov,Middleweight,Blue,35,193.04,203.2,4.03,0.480,0.1000,0.05,0.09,17,0,9,2,13,4,64,12,5,0
10,2024-08-17,B,Israel Adesanya,Dricus Du Plessis,Middleweight,Red,35,193.04,203.2,4.00,0.480,0.1000,0.05,0.12,16,0,9,1,13,3,60,11,5,0
2,2023-09-09,R,Israel Adesanya,Sean Strickland,Middleweight,Blue,34,193.04,203.2,4.00,0.480,0.1000,0.05,0.12,15,1,9,0,13,2,55,11,5,0
11,2023-04-08,B,Israel Adesanya,Alex Pereira,Middleweight,Blue,33,193.04,203.2,4.00,0.480,0.1000,0.05,0.12,14,0,9,1,12,2,53,11,4,0
3,2022-11-12,R,Israel Adesanya,Alex Pereira,Middleweight,Blue,33,193.04,203.2,4.00,0.480,0.1000,0.05,0.12,13,3,9,0,12,1,48,11,4,0
4,2022-07-02,R,Israel Adesanya,Jared Cannonier,Middleweight,Red,32,193.04,203.2,4.00,0.480,0.1000,0.05,0.12,12,2,9,0,11,1,43,11,4,0
5,2022-02-12,R,Israel Adesanya,Robert Whittaker,Middleweight,Red,32,193.04,203.2,4.00,0.480,0.1000,0.05,0.12,11,1,9,0,10,1,38,11,4,0
6,2021-06-12,R,Israel Adesanya,Marvin Vettori,Middleweight,Red,31,193.04,203.2,3.94,0.500,0.2000,0.00,0.00,10,0,9,1,9,1,33,5,4,0
12,2021-03-06,B,Israel Adesanya,Jan Blachowicz,Light Heavyweight,Red,31,193.04,203.2,3.95,0.490,0.3000,0.00,0.00,9,9,9,0,9,0,28,4,4,0
